# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import getpass
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

prev = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_prev,
        SUM(gsc_clicks) AS gsc_clicks_prev,
        AVG(gsc_avg_position) AS gsc_avg_position_prev
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date < DATE '2026-03-16'
    GROUP BY client_hash_id, content_hash_id
""").df()

target_day = con.sql(f"""
    SELECT client_hash_id, content_hash_id, gsc_clicks, gsc_impressions
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date = DATE '2026-03-31'
""").df()

full_month = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS full_march_impressions
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

prev["ctr_prev"] = prev["gsc_clicks_prev"] / prev["gsc_impressions_prev"].replace(0, np.nan)
prev["has_min_volume"] = (prev["gsc_impressions_prev"] >= 100).astype(int)

bins = [0, 3, 10, 20, 50, np.inf]
labels_ = ["top_3", "page_1", "striking", "page_3_5", "deep"]
prev["position_tier"] = pd.cut(prev["gsc_avg_position_prev"], bins=bins, labels=labels_)

feat = prev[prev["has_min_volume"] == 1].dropna(subset=["ctr_prev", "position_tier"]).copy()
feat["tier_avg_ctr_prev"] = feat.groupby("position_tier", observed=True)["ctr_prev"].transform("mean")

target_day["ctr_31"] = target_day["gsc_clicks"] / target_day["gsc_impressions"].replace(0, np.nan)
labeled = feat.merge(
    target_day[["client_hash_id", "content_hash_id", "ctr_31", "gsc_clicks"]],
    on=["client_hash_id", "content_hash_id"], how="inner"
).dropna(subset=["ctr_31"])

labeled = labeled.merge(full_month, on=["client_hash_id", "content_hash_id"], how="left")

labeled["ctr_gap"] = labeled["tier_avg_ctr_prev"] - labeled["ctr_31"]
labeled["is_underperformer"] = (labeled["ctr_gap"] > 0).astype(int)

FEATURES = ["gsc_impressions_prev", "gsc_avg_position_prev", "position_tier", "ctr_prev", "has_min_volume"]

X = pd.get_dummies(labeled[FEATURES], columns=["position_tier"])
y = labeled["is_underperformer"]

print(f"Feature frame: {X.shape[0]} rows, {X.shape[1]} columns")
print(X.head())

Feature frame: 75738 rows, 9 columns
   gsc_impressions_prev  gsc_avg_position_prev  ctr_prev  has_min_volume  \
0                 127.0               5.925563  0.000000               1   
1                1829.0               2.517877  0.003280               1   
2                 853.0               4.349245  0.000000               1   
3                 540.0               5.461846  0.001852               1   
4                 214.0               7.528073  0.000000               1   

   position_tier_top_3  position_tier_page_1  position_tier_striking  \
0                False                  True                   False   
1                 True                 False                   False   
2                False                  True                   False   
3                False                  True                   False   
4                False                  True                   False   

   position_tier_page_3_5  position_tier_deep  
0                   False

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
notes = pd.DataFrame([
    {"feature": "gsc_impressions_prev", "meaning": "trailing search impressions, days 1 to 15 of March", "missing": "0 rows, has_min_volume filter applied", "available_when": "before the scored day, built only from past dates"},
    {"feature": "gsc_avg_position_prev", "meaning": "average ranking position, days 1 to 15 of March", "missing": "0 rows after dropna", "available_when": "before the scored day, built only from past dates"},
    {"feature": "position_tier", "meaning": "position bucket computed from gsc_avg_position_prev, one hot encoded", "missing": "dropped if position missing", "available_when": "known before the scored day, computed from prior average position"},
    {"feature": "ctr_prev", "meaning": "click through rate days 1 to 15, gsc_clicks_prev over gsc_impressions_prev", "missing": "0 rows, denominator positive by has_min_volume", "available_when": "built only from clicks and impressions before the scored day"},
    {"feature": "has_min_volume", "meaning": "flag, gsc_impressions_prev >= 100", "missing": "never missing, derived from feature 1", "available_when": "same as gsc_impressions_prev, before the scored day"},
])
print(notes.to_string(index=False))

print("Missingness in raw pre filter columns")
print(prev[["gsc_impressions_prev", "gsc_avg_position_prev", "ctr_prev", "position_tier"]].isna().mean())

              feature                                                                    meaning                                        missing                                                    available_when
 gsc_impressions_prev                         trailing search impressions, days 1 to 15 of March          0 rows, has_min_volume filter applied                 before the scored day, built only from past dates
gsc_avg_position_prev                            average ranking position, days 1 to 15 of March                            0 rows after dropna                 before the scored day, built only from past dates
        position_tier       position bucket computed from gsc_avg_position_prev, one hot encoded                    dropped if position missing known before the scored day, computed from prior average position
             ctr_prev click through rate days 1 to 15, gsc_clicks_prev over gsc_impressions_prev 0 rows, denominator positive by has_min_volume      built only 

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeRegressor

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

honest_model = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X, y)
honest_score = honest_model.predict(X)
honest_p50 = precision_at_k(honest_score, y, 50)
print(f"Honest Precision@50, 5 features: {honest_p50:.3f}")

X_leak_label = X.copy()
X_leak_label["TRAP_target_day_clicks"] = labeled["gsc_clicks"].values
leak_label_model = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_leak_label, y)
leak_label_score = leak_label_model.predict(X_leak_label)
leak_label_p50 = precision_at_k(leak_label_score, y, 50)
print(f"Leaky Precision@50, target day gsc_clicks added: {leak_label_p50:.3f}")

X_leak_window = X.copy()
X_leak_window["TRAP_full_march_impressions"] = labeled["full_march_impressions"].values
leak_window_model = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_leak_window, y)
leak_window_score = leak_window_model.predict(X_leak_window)
leak_window_p50 = precision_at_k(leak_window_score, y, 50)
print(f"Leaky Precision@50, full month window overlapping label: {leak_window_p50:.3f}")

del X_leak_label, X_leak_window
print(f"Traps removed. Kept honest number: Precision@50 = {honest_p50:.3f}")

Honest Precision@50, 5 features: 0.960
Leaky Precision@50, target day gsc_clicks added: 1.000
Leaky Precision@50, full month window overlapping label: 0.960
Traps removed. Kept honest number: Precision@50 = 0.960


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = pd.DataFrame([
    {"field": "gsc_clicks target day", "why": "numerator of ctr_31, the label is_underperformer is derived from it, label derived feature"},
    {"field": "ctr_gap", "why": "this is the target proxy itself, not a feature"},
    {"field": "full month gsc_impressions", "why": "window overlaps the scored day, future information relative to the decision moment"},
    {"field": "ga4_sessions and other ga4 columns", "why": "ga4_data_available is three valued, TRUE FALSE NULL, zero fills are not reliable engagement zeros for many rows"},
    {"field": "sessions_ai", "why": "too sparse warehouse wide to support a tier level CTR comparison without noise"},
    {"field": "client_hash_id content_hash_id", "why": "pseudonymous IDs, used only for joining and grouping, never modeled"},
])
print(excluded.to_string(index=False))

                             field                                                                                                             why
             gsc_clicks target day                      numerator of ctr_31, the label is_underperformer is derived from it, label derived feature
                           ctr_gap                                                                  this is the target proxy itself, not a feature
        full month gsc_impressions                              window overlaps the scored day, future information relative to the decision moment
ga4_sessions and other ga4 columns ga4_data_available is three valued, TRUE FALSE NULL, zero fills are not reliable engagement zeros for many rows
                       sessions_ai                                  too sparse warehouse wide to support a tier level CTR comparison without noise
    client_hash_id content_hash_id                                             pseudonymous IDs, used only for joining

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.